# Agent Types in LangChain
> **Reference:** Albada, M. (2025). *Building Applications with AI Agents: Designing and Implementing Multiagent Systems*. 

---

## Overview

Each agent type embodies a **distinct approach to reasoning, planning, and action** — shaping how tasks are decomposed and executed. The choice of agent type directly influences *performance, cost, and capabilities*.

### Six Core Archetypes

| # | Agent Type | Strength | Weakness | Best Use Case |
|---|---|---|---|---|
| 1 | **Reflex** | Millisecond latency | No reasoning | Routing, lookups |
| 2 | **ReAct** | Flexible, adaptive loop | Cost / latency | Exploratory tasks |
| 3 | **Planner-Executor** | Debuggable, clear plan | Static plan | Multi-step pipelines |
| 4 | **Query-Decomposition** | Grounded retrieval | Many tool calls | Fact Q&A, research |
| 5 | **Reflection** | Early error correction | High compute | High-stakes workflows |
| 6 | **Deep Research** | Multi-stage depth | Very high cost | Literature reviews |

> 💡 **Guiding Principle:** Start with the simplest agent type that satisfies your requirements. Every step up the complexity ladder multiplies cost and latency.

## Setup — Install Dependencies

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
!pip install -q \
    langchain \
    langchain-core \
    langchain-anthropic \
    anthropic

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ""  # Replace with your key

## Common Imports & LLM Setup

All agents use **`claude-haiku-4-5`** — Anthropic's lowest-cost, fastest model — sufficient for demonstrating all agent patterns.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage
)
import json, re

# Low-cost Claude model used throughout
CHEAP_MODEL = "claude-haiku-4-5"

llm = ChatAnthropic(model=CHEAP_MODEL, temperature=0)
print(f" Using model: {CHEAP_MODEL}")

---

# 1. 🔁 Reflex Agent

A Reflex agent implements a **direct input → action mapping** with no internal reasoning trace. It follows *if-condition, then-action* rules, calling the appropriate tool immediately upon detecting predefined triggers.

Because it bypasses intermediate thought steps, it delivers responses with **minimal latency and predictable performance**.

## Architecture

```
┌─────────────────┐
│   User Input    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Rule Engine   │  ← if "weather" → call get_weather()
│  (if-then map)  │  ← if "stock"   → call get_stock()
└────────┬────────┘
         │  (no feedback loop — immediate dispatch)
         ▼
┌─────────────────┐
│    Tool Call    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│     Output      │
└─────────────────┘
```

**Key Properties:**
- ✅ Zero LLM reasoning overhead
- ✅ Fully deterministic and auditable
- ❌ Cannot handle tasks requiring multi-step reasoning
- ❌ No context beyond the immediate input

**Best Use Cases:** Keyword-based routing · Single-step data lookups · FAQ bots · Simple automations

In [ ]:
# ─── REFLEX AGENT — LangChain from Scratch ───────────────────────────────────

@tool
def get_weather(city: str) -> str:
    """Return mock weather for a city."""
    return f"Sunny, 25°C in {city.title()}"

@tool
def get_stock_price(ticker: str) -> str:
    """Return mock stock price."""
    prices = {"AAPL": 195.50, "GOOGL": 172.30, "MSFT": 415.80}
    price = prices.get(ticker.upper(), 100.00)
    return f"{ticker.upper()}: ${price}"

@tool
def get_time(timezone: str) -> str:
    """Return mock current time for a timezone."""
    return f"🕐 Current time in {timezone}: 14:32 PST"

# Rule-based router — pure if-then, NO LLM reasoning involved
RULES = {
    "weather": (get_weather,  "city"),
    "stock":   (get_stock_price, "ticker"),
    "time":    (get_time,     "timezone"),
}

def reflex_agent(user_input: str) -> str:
    """Direct input → action mapping without LLM reasoning."""
    user_lower = user_input.lower()

    for keyword, (tool_fn, arg_key) in RULES.items():
        if keyword in user_lower:
            # Naive argument extraction (real impl: regex / NER)
            parts = user_lower.split(keyword)
            arg   = parts[-1].strip(" ?.,!").split()[-1] if parts[-1].strip() else "unknown"
            print(tool_fn)
            return tool_fn.invoke({arg_key: arg})

    # Fallback: direct LLM call (no tool loop)
    return llm.invoke([HumanMessage(content=user_input)]).content

# ─── Test ─────────────────────────────────────────────────────────────────────
queries = [
    "What is the weather in London?",
    "Get stock price for AAPL",
    "What time is it in Tokyo?",
    "Tell me a fun fact",  # fallback
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {reflex_agent(q)}\n")

---

# 2. 🔄 ReAct Agent



**Re**asoning + **Act**ion interleaved in an iterative loop. The model generates a *thought*, selects and invokes a tool, observes the result, and repeats as needed. This pattern enables the agent to break complex tasks into manageable steps, **updating its plan based on intermediate observations**.

## Architecture

```
┌─────────────────┐
│   User Input    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│     Thought     │ ← LLM reasons about what to do next
│   (LLM call)    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│     Action      │ ← LLM selects and calls a tool
│   (Tool Call)   │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Observation   │ ← Tool returns result
│  (Tool Output)  │
└────────┬────────┘
         │
    ─────┴──────────────────────────────────┐
    │  iterate (more steps needed?)         │
    ▼                                       │
┌─────────────────┐                    back to Thought
│  Final Answer   │ ← no more tool calls needed
└─────────────────┘
```

**Key Properties:**
- ✅ Flexible, on-the-fly adaptation
- ✅ Transparent chain-of-thought (great for debugging)
- ✅ Handles multi-source aggregation
- ❌ Higher latency per query
- ❌ Increased token/API cost

**Best Use Cases:** Dynamic data analysis · Multi-source aggregation · Troubleshooting · Exploratory workflows

In [ ]:
# ─── ReAct AGENT — LangChain from Scratch ────────────────────────────────────

@tool
def search_web(query: str) -> str:
    """Search the web for factual information."""
    mock_db = {
        "python version":   "Python 3.13 was released in October 2024.",
        "openai founded":   "OpenAI was founded in December 2015.",
        "langchain":        "LangChain is an open-source framework for building LLM apps, founded in 2022.",
    }
    for k, v in mock_db.items():
        if k in query.lower():
            return v
    return f"[Web result for '{query}']: Found 3 relevant articles."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        # Safe eval for basic math
        allowed = set('0123456789+-*/()., ')
        if all(c in allowed for c in expression):
            return str(round(eval(expression), 4))
        return "Error: unsafe expression"
    except Exception as e:
        return f"Error: {e}"

tools     = [search_web, calculator]
tools_map = {t.name: t for t in tools}
llm_react = ChatAnthropic(model=CHEAP_MODEL, temperature=0).bind_tools(tools)

def react_agent(user_input: str, max_steps: int = 6, verbose: bool = True) -> str:
    """ReAct loop: Thought → Action → Observation → repeat."""
    messages = [HumanMessage(content=user_input)]
    step = 0

    while step < max_steps:
        response = llm_react.invoke(messages)
        messages.append(response)
        step += 1

        # ── No tool calls → final answer ──────────────────────────────────────
        if not response.tool_calls:
            if verbose: print(f"Final Answer (step {step}): {response.content}")
            return response.content

        # ── Execute all tool calls (Act + Observe) ────────────────────────────
        for tc in response.tool_calls:
            if verbose: print(f"  🔧 Step {step} | Tool: {tc['name']} | Args: {tc['args']}")
            result = tools_map[tc["name"]].invoke(tc["args"])
            if verbose: print(f"     📋 Observation: {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return "Max steps reached without final answer."

# ─── Test ─────────────────────────────────────────────────────────────────────

react_agent("When was LangChain founded and how many years ago was that from 2025?")

---

# 3. 📋 Planner-Executor Agent

Splits a task into two **distinct phases**:
1. **Planning** — the model generates a multi-step plan
2. **Execution** — each planned step is carried out via tool calls

This separation lets the planner focus on *long-horizon reasoning* while executors invoke only the necessary tools, reducing redundant LLM calls.

## Architecture

```
┌─────────────────────┐
│     User Goal       │
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐
│    PLANNER LLM      │  ← Generates ordered list of steps
│  (large / capable)  │    [{step:1, desc:"...", tool:"..."},
└──────────┬──────────┘     {step:2, ...}, ...]
           │
           ▼
┌─────────────────────┐
│    Step i of N      │  ← Iterates through plan
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐
│   EXECUTOR LLM      │  ← Carries out one step
│  (small / cheap)    │    (tool call or direct LLM)
└──────────┬──────────┘
           │
    ───────┴──────────────────────────┐
    │  next step                      │
    ▼                                 │
┌─────────────────────┐          back to Step i
│    Final Output     │  ← all steps complete
└─────────────────────┘
```

**Key Properties:**
- ✅ Clear task decomposition — each step is inspectable
- ✅ Debuggable — you can see exactly where failures occur
- ✅ Cost-efficient — cheap model executes, expensive model only plans
- ❌ Static plan may not adapt if mid-task conditions change
- ❌ Extra LLM call for planning phase

**Best Use Cases:** Report generation · Data pipelines · Complex multi-step workflows · Automated research tasks

In [ ]:
# ─── PLANNER-EXECUTOR AGENT — Fixed ──────────────────────────────────────────

@tool
def fetch_data(source: str) -> str:
    """Fetch data from a given source."""
    mock = {
        "sales":     "Q1=$1.0M, Q2=$1.4M, Q3=$1.8M, Q4=$2.1M",
        "customers": "Total: 4,200 | New: 850 | Churned: 120",
        "expenses":  "Payroll=$600K, Marketing=$200K, Infra=$80K",
    }
    for k, v in mock.items():
        if k in source.lower():
            return v
    return f"[Data from '{source}']: 500 records loaded."

@tool
def analyze(data: str) -> str:
    """Analyze data and return key insights."""
    return f"Analysis of '{data[:40]}': Growth trend detected, 27% YoY increase."

@tool
def write_report(content: str) -> str:
    """Compile and save a final report."""
    return f"📄 Report saved ({len(content)} chars). Preview: {content[:80]}..."

exec_tools     = [fetch_data, analyze, write_report]
exec_tools_map = {t.name: t for t in exec_tools}

planner_llm  = ChatAnthropic(model=CHEAP_MODEL, temperature=0)
executor_llm = ChatAnthropic(model=CHEAP_MODEL, temperature=0).bind_tools(exec_tools)

# ── FIX 1: Explicit arg schemas in the prompt ─────────────────────────────────
PLAN_PROMPT = """Break the user goal into an ordered JSON list of steps.
Each step MUST follow this EXACT schema:
  {"step": int, "description": str, "tool": str or null, "args": dict or null}

Available tools and their EXACT argument names (use these exactly):
  - fetch_data(source: str)       → source must be one of: "sales", "customers", "expenses"
  - analyze(data: str)            → data is a string description of what to analyze
  - write_report(content: str)    → content is the full report text to save

Rules:
  - "args" keys must EXACTLY match the parameter names shown above
  - If no tool is needed, set "tool": null and "args": null
  - Return ONLY a valid JSON array. No markdown, no explanation."""


def planner(goal: str) -> list:
    resp = planner_llm.invoke([
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=goal)
    ])
    raw = re.sub(r"```json|```", "", resp.content).strip()
    return json.loads(raw)


# ── FIX 2: Safe executor with arg validation fallback ─────────────────────────
TOOL_ARG_MAP = {
    "fetch_data":   "source",
    "analyze":      "data",
    "write_report": "content",
}

def safe_args(tool_name: str, raw_args: dict) -> dict:
    """Remap any hallucinated arg names to the correct single argument."""
    expected_key = TOOL_ARG_MAP.get(tool_name)
    if not expected_key:
        return raw_args
    # Already correct
    if expected_key in raw_args:
        return raw_args
    # Remap: take the first value from whatever keys the LLM made up
    if raw_args:
        first_value = next(iter(raw_args.values()))
        print(f"Remapped args {raw_args} → {{'{expected_key}': '{first_value}'}}")
        return {expected_key: str(first_value)}
    return {expected_key: tool_name}  # last-resort fallback


def executor(step: dict) -> str:
    tool_name = step.get("tool")
    if tool_name and tool_name in exec_tools_map:
        corrected_args = safe_args(tool_name, step.get("args") or {})
        return exec_tools_map[tool_name].invoke(corrected_args)
    # Generic LLM step for non-tool actions
    resp = executor_llm.invoke([HumanMessage(content=step["description"])])
    return resp.content


def planner_executor_agent(goal: str) -> list:
    print(f"Goal: {goal}\n")
    plan = planner(goal)
    print(f"Plan ({len(plan)} steps):")
    for s in plan:
        print(f"  \n Step {s['step']}: {s['description']} | tool={s.get('tool')} args={s.get('args')}")
    print()

    results = []
    for step in plan:
        print(f"Executing step {step['step']}: {step['description']}")
        result = executor(step)
        print(f"{result}\n")
        results.append({"step": step["step"], "description": step["description"], "result": result})
    return results

# ─── Test ─────────────────────────────────────────────────────────────────────
planner_executor_agent("Generate a quarterly revenue trend report with analysis.")


---

# 4. 🔍 Query-Decomposition Agent


Tackles a complex question by **iteratively breaking it into sub-questions**, invoking search tools for each, then synthesizing a final answer. This pattern — often called *"self-ask with search"* — ensures every fact is grounded in tool output before composing the response.

## Architecture

```
┌──────────────────────────┐
│     Complex Question     │
└────────────┬─────────────┘
             │
             ▼
┌──────────────────────────┐
│  Decompose (LLM self-ask)│  ← "What follow-up question do I need?"
└────────────┬─────────────┘
             │
             ▼
┌──────────────────────────┐
│     Sub-question i       │  e.g. "What was Einstein's lifespan?"
└────────────┬─────────────┘
             │
             ▼
┌──────────────────────────┐
│    Search / Tool Call    │  → "Einstein lived 76 years (1879-1955)"
└────────────┬─────────────┘
             │
             ▼
┌──────────────────────────┐
│     Partial Answer       │
└────────────┬─────────────┘
             │
    ─────────┴─────────────────────────────────────┐
    │  more sub-questions needed?                   │
    ▼                                               │
┌──────────────────────────┐                back to Decompose
│   Synthesize Final Answer│  ← all sub-questions resolved
└──────────────────────────┘
```

**Classic Example (from the book):**
- ❓ Q: *"Who lived longer, Einstein or Turing?"*
- 🔎 Sub-Q1: *"What was Einstein's lifespan?"* → 76 years
- 🔎 Sub-Q2: *"What was Turing's lifespan?"* → 41 years  
- ✅ Synthesis: *"Einstein (76 yrs) outlived Turing (41 yrs)"*

**Key Properties:**
- ✅ Every fact grounded in tool output
- ✅ Handles multi-hop reasoning
- ❌ Multiple sequential tool calls — higher latency
- ❌ Can over-decompose simple questions

**Best Use Cases:** Research Q&A · Fact-checking · Multi-hop knowledge retrieval

In [ ]:
# ─── QUERY-DECOMPOSITION AGENT — LangChain from Scratch ──────────────────────

@tool
def search(query: str) -> str:
    """Search for factual information."""
    knowledge = {
        "einstein lifespan":    "Albert Einstein lived 76 years (born 1879, died 1955).",
        "turing lifespan":      "Alan Turing lived 41 years (born 1912, died 1954).",
        "newton lifespan":      "Isaac Newton lived 84 years (born 1643, died 1727).",
        "curie lifespan":       "Marie Curie lived 66 years (born 1867, died 1934).",
        "einstein nobel":       "Einstein won the Nobel Prize in Physics in 1921.",
        "turing award":         "Turing is considered the father of computer science.",
        "einstein born":        "Einstein was born in Ulm, Germany in 1879.",
        "turing born":          "Turing was born in London, England in 1912.",
    }
    query_lower = query.lower()
    for k, v in knowledge.items():
        if k in query_lower:
            return v
    return f"[Search result for '{query}']: Information found in 2 sources."

SELF_ASK_PROMPT = """You are a self-asking research agent.
Given a question, respond with EXACTLY one of:

  Follow-up: <a specific sub-question you need answered>
  Answer: <the final synthesized answer>

Use Follow-up when you need more information.
Use Answer ONLY when you have enough info to answer the original question fully.
Be concise. Do not add explanations beyond the format above."""

qd_llm = ChatAnthropic(model=CHEAP_MODEL, temperature=0)

def query_decomp_agent(question: str, max_steps: int = 8, verbose: bool = True) -> str:
    """Iteratively decompose a question into sub-questions and synthesize the answer."""
    messages = [
        SystemMessage(content=SELF_ASK_PROMPT),
        HumanMessage(content=f"Original question: {question}")
    ]
    gathered_facts = []

    for step in range(max_steps):
        resp = qd_llm.invoke(messages)
        text = resp.content.strip()

        if verbose: print(f"Agent [{step+1}]: {text}")

        # ── Final answer reached ──────────────────────────────────────────────
        if text.startswith("Answer:"):
            return text[len("Answer:"):].strip()

        # ── Follow-up sub-question → call search ──────────────────────────────
        if text.startswith("Follow-up:"):
            sub_q  = text[len("Follow-up:"):].strip()
            result = search.invoke({"query": sub_q})
            gathered_facts.append(f"Q: {sub_q} | A: {result}")
            if verbose: print(f"Search('{sub_q}'): {result}")

            # Feed back all gathered facts to help reach a final answer
            facts_str = "\n".join(gathered_facts)
            messages  = [
                SystemMessage(content=SELF_ASK_PROMPT),
                HumanMessage(content=(
                    f"Original question: {question}\n\n"
                    f"Facts gathered so far:\n{facts_str}\n\n"
                    f"Continue: do you need more info or can you answer now?"
                ))
            ]

    return "Could not resolve within max steps."

In [ ]:
print("Q: Who lived longer, Einstein or Turing?")
ans = query_decomp_agent("Who lived longer, Einstein or Turing?")
print(f"Final: {ans}")

---

# 5.  Reflection Agent

Extends the ReAct paradigm by adding a **self-critique step** after each action. The agent continuously grounds its reasoning in *goal-state reflections*, measuring current state against the intended outcome and adjusting the plan when misalignments arise. This is exemplified by the **ReflAct framework**.

> *"Reflection prompts encourage the model to critique its own chain of thought, correct logical errors, and reinforce successful strategies, effectively simulating human-style self-assessment."* — Albada (2025)

## Architecture

```
┌────────────────────────┐
│      User Input        │
└───────────┬────────────┘
            │
            ▼
┌────────────────────────┐
│   Thought (Actor LLM)  │  ← Reasons about next action
└───────────┬────────────┘
            │
            ▼
┌────────────────────────┐
│   Action (Tool Call)   │  ← Invokes a tool
└───────────┬────────────┘
            │
            ▼
┌────────────────────────┐
│      Observation       │  ← Tool returns result
└───────────┬────────────┘
            │
            ▼
┌────────────────────────┐
│  REFLECT (Critic LLM)  │  ← "Was this correct? Any errors?"
│  (self-critique step)  │     "Am I aligned with the goal?"
└───────────┬────────────┘
            │
    ────────┴──────────────────────────────────────┐
    │  CONTINUE: error found, revise plan           │
    ▼                                               │
┌────────────────────────┐                  back to Thought
│     Final Answer       │  ← DONE: no errors, goal met
└────────────────────────┘
```

**Key Properties:**
- ✅ Detects and corrects errors before they cascade
- ✅ Ideal for high-stakes, safety-critical workflows
- ✅ Dual-LLM: actor generates, critic evaluates
- ❌ 2× LLM calls per step → higher cost and latency

**Best Use Cases:** Financial transaction orchestration · Medical diagnosis support · Critical incident response · Code generation with verification

In [ ]:
# ─── REFLECTION AGENT — Fixed ─────────────────────────────────────────────────

REFLECT_SYSTEM = """You are a critical evaluator reviewing an AI agent's action.
Given the original goal, action taken, and observation received, respond with EXACTLY:
  DONE: <final synthesized answer>    ← goal is fully and correctly achieved
  CONTINUE: <specific correction>     ← error found or goal not yet met
Be critical but fair."""

def reflection_agent(goal: str, max_steps: int = 5, verbose: bool = True) -> str:
    """ReAct loop with self-critique — Anthropic tool_use/tool_result pair safe."""
    messages = [HumanMessage(content=goal)]

    for step in range(max_steps):
        # ── Actor step ────────────────────────────────────────────────────────
        response = actor_llm.invoke(messages)
        messages.append(response)

        # ── Case 1: No tool calls — direct LLM answer ─────────────────────────
        if not response.tool_calls:
            action_desc = response.content
            observation = "(no tool called)"

            if verbose:
                print(f"\n[Step {step+1}]")
                print(f"  🎬 Action:      {action_desc}")
                print(f"  📋 Observation: {observation}")

        # ── Case 2: Tool calls — MUST append ALL tool_results before anything else
        else:
            action_desc = ""
            observation = ""

            for tc in response.tool_calls:
                obs = actor_tools_map[tc["name"]].invoke(tc["args"])
                # ⚠️ Append tool_result IMMEDIATELY after tool_use — no other
                #    messages in between or Anthropic raises BadRequestError
                messages.append(
                    ToolMessage(content=str(obs), tool_call_id=tc["id"])
                )
                action_desc += f"Called {tc['name']}({tc['args']}) | "
                observation += str(obs) + " | "

            action_desc = action_desc.rstrip(" | ")
            observation = observation.rstrip(" | ")

            if verbose:
                print(f"\n[Step {step+1}]")
                print(f"  🎬 Action:      {action_desc}")
                print(f"  📋 Observation: {observation}")

        # ── Reflect AFTER all tool_results are appended ───────────────────────
        verdict = critic_llm.invoke([
            SystemMessage(content=REFLECT_SYSTEM),
            HumanMessage(content=(
                f"Goal: {goal}\n"
                f"Action: {action_desc}\n"
                f"Observation: {observation}"
            ))
        ]).content.strip()

        if verbose:
            print(f"  🪞 Reflect:     {verdict}")

        # ── DONE → return final answer ─────────────────────────────────────────
        if verdict.startswith("DONE:"):
            return verdict[5:].strip()

        # ── CONTINUE → inject correction as HumanMessage AFTER tool_results ───
        # Safe here because all tool_results are already appended above
        correction = verdict[len("CONTINUE:"):].strip() \
                     if verdict.startswith("CONTINUE:") else verdict
        messages.append(
            HumanMessage(content=f"Reflection feedback: {correction}. Please revise.")
        )

    return messages[-1].content if messages else "Max steps reached."

In [ ]:
print("Goal: Analyze the sales dataset and report any anomalies.")
result = reflection_agent(
    "Analyze the sales dataset and report any anomalies with validation."
)
print(f"Final Answer: {result}")

---

# 6. 🔬 Deep Research Agent


The most powerful and complex agent type. **Combines all prior patterns** into a single pipeline for tackling open-ended, multi-stage investigations requiring extensive knowledge gathering, hypothesis testing, and synthesis.

> *"Deep research agents combine multiple patterns: a planner-executor phase to chart research workflows; query-decomposition to break down big questions into targeted searches; and ReAct loops to iteratively refine hypotheses based on new findings."* — Albada (2025)

## Architecture

```
┌──────────────────────────────────────────┐
│            Research Goal                 │
└─────────────────┬────────────────────────┘
                  │
                  ▼
┌──────────────────────────────────────────┐
│  PHASE 1: PLANNER LLM                    │
│  → Identify key subtopics                │
│  → Generate research agenda              │
│  Output: [{subtopic, queries[]}]         │
└─────────────────┬────────────────────────┘
                  │
         ┌────────┴──────────────┐
         ▼                       ▼
┌─────────────────┐   ┌─────────────────┐
│   Subtopic A    │   │   Subtopic B    │  (parallel possible)
└────────┬────────┘   └────────┬────────┘
         │                     │
         ▼                     ▼
┌──────────────────────────────────────────┐
│  PHASE 2: DECOMPOSER                     │
│  → Break each subtopic into queries      │
└─────────────────┬────────────────────────┘
                  │
                  ▼
┌──────────────────────────────────────────┐
│  PHASE 3: ReAct LOOP                     │
│  → academic_search / web_search tools    │
│  → Thought → Action → Observation loop  │
└─────────────────┬────────────────────────┘
                  │
                  ▼
┌──────────────────────────────────────────┐
│  PHASE 4: REFLECTION                     │
│  → Quality check on gathered findings   │
│  → SYNTHESIZE or CONTINUE?              │
└─────────────────┬────────────────────────┘
                  │ (loop back if more research needed)
                  ▼
┌──────────────────────────────────────────┐
│  PHASE 5: SYNTHESIZER                    │
│  → Compile all findings into report      │
└──────────────────────────────────────────┘
```

**Key Properties:**
- ✅ Handles high-complexity, multi-stage investigations
- ✅ Adaptive — research direction adjusts as evidence emerges
- ✅ Transparent — explicit plans and decomposition steps are auditable
- ❌ Very high compute cost (many LLM calls)
- ❌ High latency — each layer adds delay
- ❌ Fragile if external data sources are unavailable

**Best Use Cases:** Academic literature surveys · Technical due diligence · Competitive intelligence · Scientific discovery workflows

In [ ]:
# ─── DEEP RESEARCH AGENT — LangChain from Scratch ────────────────────────────

@tool
def academic_search(query: str) -> str:
    """Search academic papers and journals."""
    mock = {
        "llm productivity":    "5 studies found. Meta-analysis: 37% avg productivity gain for developers.",
        "code generation":     "GitHub Copilot study: 55% faster task completion (Ziegler et al., 2022).",
        "software quality":    "Mixed results: LLM tools improve speed but require careful review for bugs.",
        "developer adoption":  "70% of Fortune 500 devs now use AI coding assistants (survey, 2024).",
    }
    for k, v in mock.items():
        if k in query.lower():
            return f"[Academic] {v}"
    return f"[Academic] '{query}': 3 relevant papers found. Key themes: impact, adoption, challenges."

@tool
def web_search_research(query: str) -> str:
    """Search the web for recent news and industry reports."""
    mock = {
        "market size":      "AI coding tools market: $4.7B in 2024, projected $22B by 2028.",
        "recent trends":    "2025 trend: Agentic coding (autonomous PR creation, test generation).",
        "industry report":  "McKinsey 2024: Generative AI could automate 30% of developer tasks.",
    }
    for k, v in mock.items():
        if k in query.lower():
            return f"[Web] {v}"
    return f"[Web] '{query}': Recent articles show strong adoption and growing ecosystem."

@tool
def summarize_findings(text: str) -> str:
    """Summarize a collection of research findings."""
    return f"Summary ({len(text.split())} words condensed): Key finding — significant positive impact with nuanced trade-offs."

research_tools     = [academic_search, web_search_research, summarize_findings]
research_tools_map = {t.name: t for t in research_tools}

research_llm      = ChatAnthropic(model=CHEAP_MODEL, temperature=0)
research_llm_bind = ChatAnthropic(model=CHEAP_MODEL, temperature=0).bind_tools(research_tools)

PLAN_SYSTEM = """Create a research plan as a JSON list. No markdown, no explanation.
Format: [{"subtopic": "str", "queries": ["query1", "query2"]}]
Generate 2-3 subtopics with 2 queries each."""

REFLECT_SYSTEM_DR = """Review gathered research findings.
Respond ONLY with:
  SYNTHESIZE  ← if enough information is gathered for a comprehensive report
  CONTINUE: <specific missing angle to research>  ← if critical gaps remain"""

def deep_research_agent(research_goal: str, verbose: bool = True) -> str:
    print(f"Research Goal: {research_goal}\n{'='*60}")
    all_findings = []

    # ── PHASE 1: Plan ─────────────────────────────────────────────────────────
    plan_resp = research_llm.invoke([
        SystemMessage(content=PLAN_SYSTEM),
        HumanMessage(content=research_goal)
    ])
    plan = json.loads(re.sub(r"```json|```", "", plan_resp.content).strip())
    if verbose:
        print(f"Research Plan ({len(plan)} subtopics):")
        for p in plan: print(f"   • {p['subtopic']}")
        print()

    # ── PHASE 2 & 3: Decompose + ReAct per subtopic ───────────────────────────
    for subtopic in plan:
        if verbose: print(f"\Subtopic: {subtopic['subtopic']}")

        for query in subtopic["queries"]:
            if verbose: print(f"Query: {query}")
            messages = [HumanMessage(content=f"Research this specific query: {query}")]

            # ReAct sub-loop (max 3 steps per query)
            for react_step in range(3):
                resp = research_llm_bind.invoke(messages)
                messages.append(resp)

                if not resp.tool_calls:
                    all_findings.append(f"[{subtopic['subtopic']}] {resp.content}")
                    if verbose: print(f"{resp.content[:80]}...")
                    break

                for tc in resp.tool_calls:
                    obs = research_tools_map[tc["name"]].invoke(tc["args"])
                    all_findings.append(f"[{subtopic['subtopic']}] {obs}")
                    if verbose: print(f"{tc['name']}: {obs}")
                    messages.append(ToolMessage(content=str(obs), tool_call_id=tc["id"]))

        # ── PHASE 4: Reflect ───────────────────────────────────────────────────
        recent = "\n".join(all_findings[-4:])
        verdict = research_llm.invoke([
            SystemMessage(content=REFLECT_SYSTEM_DR),
            HumanMessage(content=f"Goal: {research_goal}\nFindings so far:\n{recent}")
        ]).content.strip()

        if verbose: print(f"\n  🪞 Reflect: {verdict[:80]}")

        if verdict.startswith("CONTINUE:"):
            extra_query = verdict[9:].strip()
            extra_result = web_search_research.invoke({"query": extra_query})
            all_findings.append(f"[Additional] {extra_result}")
            if verbose: print(f"Added: {extra_result}")

    # ── PHASE 5: Synthesize ────────────────────────────────────────────────────
    if verbose: print(f"\n{'='*60} Synthesizing {len(all_findings)} findings...")

    synthesis = research_llm.invoke([
        SystemMessage(content=(
            "You are a research synthesizer. Compile the findings into a "
            "structured report with: Executive Summary, Key Findings (3-5 bullets), "
            "and Conclusion."
        )),
        HumanMessage(content=(
            f"Research Goal: {research_goal}\n\n"
            f"Findings:\n" + "\n".join(all_findings)
        ))
    ])
    return synthesis.content

In [ ]:
report = deep_research_agent("Impact of LLMs on software engineering productivity")
print(f"FINAL REPORT:\n{'='*60}\n{report}")

---

# 📊 Comparison Summary

*(Adapted from Albada, 2025, Table 5-1, p. 93)*

| Agent Type | LLM Calls | Latency | Cost | Reasoning Depth | Best Use Case |
|---|:---:|:---:|:---:|:---:|---|
| **Reflex** | 0–1 | ⚡ μs | 💲 | None | Routing, simple lookups |
| **ReAct** | n | 🔶 Medium | 💲💲 | Medium | Exploratory workflows |
| **Planner-Executor** | 1+n | 🔶 Medium | 💲💲 | High | Multi-step pipelines |
| **Query-Decomp.** | n | 🔶 Medium | 💲💲 | High | Research, fact Q&A |
| **Reflection** | 2n | 🔴 High | 💲💲💲 | Very High | High-stakes tasks |
| **Deep Research** | ≫n | 🔴🔴 Very High | 💲💲💲💲 | Maximum | Literature reviews, due diligence |

---

## 🧭 Selection Heuristic

```
Start Here
     │
     ▼
Is the task a simple if-then lookup?  ──YES──► Reflex Agent
     │ NO
     ▼
Does it need dynamic tool use?         ──YES──► ReAct Agent
     │ NO
     ▼
Is it a complex multi-step workflow?   ──YES──► Planner-Executor
     │ NO
     ▼
Does it need multi-hop retrieval?      ──YES──► Query-Decomposition
     │ NO
     ▼
Is it high-stakes (errors are costly)? ──YES──► Reflection Agent
     │ NO
     ▼
Is it an open-ended deep investigation?──YES──► Deep Research Agent
```

> 💡 **Key Takeaway:** Always start with the **simplest** agent type. Escalate complexity only when task requirements demand it — each step up multiplies cost, latency, and debugging effort.

---
*Reference: Albada, M. (2025). Building Applications with AI Agents. O'Reilly Media.*